# Irish Property Prices - Data Cleaning

Notebook 2 of 4. Takes the raw PPR and BER files inspected in notebook 01 and produces cleaned datasets for analysis and modelling. Every row removed is logged with a reason, and the full drop log is printed at the end.

Outputs are written to data/processed/.

## 1. Setup

A running log records every filtering decision. Rather than cleaning silently, each stage reports how many rows it removes and why, which produces an auditable summary at the end of the notebook.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

RAW = Path("../Data/Raw")
PROCESSED = Path("../Data/Processed")
PROCESSED.mkdir(parents=True, exist_ok=True)

# a running log of every row dropped, printed at the end
drop_log = []

def log_drop(stage, before, after):
    dropped = before - after
    drop_log.append({
        "stage": stage,
        "dropped": dropped,
        "remaining": after,
        "pct_of_original": round(dropped / before * 100, 2)
    })
    print(f"{stage:45s} dropped {dropped:>7,}  remaining {after:>8,}")

In [2]:
ppr = pd.read_csv(RAW / "ppr_raw.csv", encoding="latin-1")
n_original = len(ppr)

print(f"Loaded {n_original:,} rows")
print(f"Columns: {list(ppr.columns)}")

Loaded 799,067 rows
Columns: ['Date of Sale (dd/mm/yyyy)', 'Address', 'County', 'Eircode', 'Price (\x80)', 'Not Full Market Price', 'VAT Exclusive', 'Description of Property', 'Property Size Description']


C:\Users\heffo\AppData\Local\Temp\ipykernel_2164\1920956887.py:1: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  ppr = pd.read_csv(RAW / "ppr_raw.csv", encoding="latin-1")


## 2. Property Price Register

### 2.1 Column naming and type parsing

The raw column names contain spaces and the mangled euro symbol, making them awkward to work with. Columns are renamed to snake_case and the price and date fields parsed to numeric and datetime types respectively. Price arrives as text with a latin-1 euro symbol and comma thousand separators.

In [3]:
# Renaming columns

ppr = ppr.rename(columns={
    "Date of Sale (dd/mm/yyyy)":  "date",
    "Address":                    "address",
    "County":                     "county",
    "Eircode":                    "eircode",
    "Price (\x80)":               "price_raw",
    "Not Full Market Price":      "not_full_market",
    "VAT Exclusive":              "vat_exclusive",
    "Description of Property":    "description",
    "Property Size Description":  "size_description",
})

print(list(ppr.columns))

['date', 'address', 'county', 'eircode', 'price_raw', 'not_full_market', 'vat_exclusive', 'description', 'size_description']


In [4]:
# Parse price and date 

ppr["price"] = (ppr["price_raw"]
                .str.replace("\x80", "", regex=False)
                .str.replace(",", "", regex=False)
                .str.strip()
                .astype(float))

ppr["date"] = pd.to_datetime(ppr["date"], format="%d/%m/%Y")
ppr["year"] = ppr["date"].dt.year
ppr["month"] = ppr["date"].dt.to_period("M")

print(f"Date range: {ppr['date'].min().date()} to {ppr['date'].max().date()}")
print(f"\nPrice summary:")
print(ppr["price"].describe().apply(lambda v: f"{v:,.0f}").to_string())

Date range: 2010-01-01 to 2026-07-31

Price summary:
count        799,067
mean         319,174
std        1,234,377
min            5,001
25%          145,000
50%          243,500
75%          365,000
max      387,665,198


In [5]:
print("Ten highest:")
print(ppr.nlargest(10, "price")[["date", "address", "county", "price"]].to_string())

print("\nTen lowest:")
print(ppr.nsmallest(10, "price")[["date", "address", "county", "price"]].to_string())

Ten highest:
             date                                                                      address   county         price
690730 2024-10-17                                  24 Tinakilly Grove, Tinakilly Park, Rathnew  Wicklow  3.876652e+08
586434 2023-02-10                                     O'Devaney Gardens, Arbour Hill, Dublin 7   Dublin  2.250000e+08
771088 2026-01-27                                                         Montpelier, Dublin 7   Dublin  2.250000e+08
706188 2024-12-23                                       Cooper Square, Seven Mills, Clonburris   Dublin  2.219427e+08
780814 2026-03-31                                            BLOCK A  B AND C, NEWMARKET YARDS   Dublin  1.893925e+08
431243 2020-07-17  Apartments 1 - 186 Cheevers Court, Apartments 1-182 Haliday House, Cualanor   Dublin  1.823789e+08
473413 2021-04-15                                          8th Lock, Ratoath Road, Pelletstown   Dublin  1.701428e+08
749598 2025-09-30                     Block

In [6]:
print("Price distribution tails:")
for q in [0.0001, 0.001, 0.005, 0.01, 0.05, 0.5, 0.95, 0.99, 0.995, 0.999, 0.9999]:
    print(f"  {q:>8.2%}  {ppr['price'].quantile(q):>15,.0f}")

Price distribution tails:
     0.01%            5,714
     0.10%            8,000
     0.50%           16,667
     1.00%           24,000
     5.00%           50,100
    50.00%          243,500
    95.00%          700,000
    99.00%        1,412,500
    99.50%        2,000,000
    99.90%        6,239,804
    99.99%       50,690,876


In [7]:
extremes = ppr[(ppr["price"] < 20_000) | (ppr["price"] > 2_000_000)]
print(f"Rows outside 20k - 2m: {len(extremes):,}  ({len(extremes)/len(ppr)*100:.2f}%)")
print(f"\nOf which flagged not-full-market:")
print(extremes["not_full_market"].value_counts().to_string())

print(f"\nSplit by tail:")
low = ppr[ppr["price"] < 20_000]
high = ppr[ppr["price"] > 2_000_000]
print(f"  below 20k:  {len(low):,}  ({(low['not_full_market']=='Yes').sum():,} flagged)")
print(f"  above 2m:   {len(high):,}  ({(high['not_full_market']=='Yes').sum():,} flagged)")

Rows outside 20k - 2m: 8,836  (1.11%)

Of which flagged not-full-market:
not_full_market
No     7603
Yes    1233

Split by tail:
  below 20k:  4,888  (1,029 flagged)
  above 2m:   3,948  (204 flagged)


### 2.2 Category harmonisation

The property description field contains Irish-language equivalents of the English categories, plus mojibake variants where the character encoding has failed at source. These represent 49 sales across five distinct strings but describe only two underlying categories. Matching on the substrings "Nua" and "Ath" folds all variants onto New and Second-Hand.

In [8]:
print("Before mapping:")
print(ppr["description"].value_counts(dropna=False).to_string())

Before mapping:
description
Second-Hand Dwelling house /Apartment    656283
New Dwelling house /Apartment            142735
Teach/Árasán Cónaithe Atháimhe               45
Teach/Árasán Cónaithe Nua                     3
Teach/?ras?n C?naithe Nua                     1


In [9]:
# anything containing "Nua" (new) or the mojibake equivalent is a new dwelling,
# anything containing "Atháimhe" / "Ath" is second-hand

def map_description(val):
    v = str(val)
    if "New Dwelling" in v or "Nua" in v:
        return "New"
    if "Second-Hand" in v or "Ath" in v or "th\u00e1imhe" in v:
        return "Second-Hand"
    return np.nan

ppr["property_type"] = ppr["description"].apply(map_description)

print("After mapping:")
print(ppr["property_type"].value_counts(dropna=False).to_string())

After mapping:
property_type
Second-Hand    656328
New            142739


### 2.3 VAT adjustment

New residential property prices are filed in the PPR excluding VAT, charged at 13.5% in Ireland. Without adjustment, new build prices are systematically understated by that margin relative to second-hand sales.

The adjustment keys off the VAT Exclusive flag rather than the property description, since 2,365 sales described as new dwellings are not flagged VAT-exclusive. Their price distribution sits well below flagged new builds rather than above, so these are unlikely to be gross-priced new sales and are more plausibly non-standard transactions such as local authority or affordable housing transfers. 1,901 survive filtering and are left unadjusted.

In [10]:
# new residential property prices in the PPR are filed excluding VAT
# Irish VAT on new residential property is 13.5%

VAT_RATE = 0.135

ppr["price_incl_vat"] = np.where(
    ppr["vat_exclusive"] == "Yes",
    ppr["price"] * (1 + VAT_RATE),
    ppr["price"]
)

n_adjusted = (ppr["vat_exclusive"] == "Yes").sum()
print(f"VAT adjustment applied to {n_adjusted:,} sales")

comparison = ppr.groupby("vat_exclusive").agg(
    n=("price", "size"),
    median_before=("price", "median"),
    median_after=("price_incl_vat", "median"),
)
print(f"\n{comparison.to_string()}")

VAT adjustment applied to 140,374 sales

                    n  median_before  median_after
vat_exclusive                                     
No             658693       226000.0    226000.000
Yes            140374       308369.0    349998.815


The adjusted new build median resolves to €349,999, essentially exactly €350,000. This is a useful confirmation that the rate and direction are correct - new builds cluster at round-number prices, and stripping VAT at source produces the awkward decimals seen in the raw data.

In [11]:
print(pd.crosstab(ppr["property_type"], ppr["vat_exclusive"]))

vat_exclusive      No     Yes
property_type                
New              2365  140374
Second-Hand    656328       0


In [12]:
new_only = ppr[ppr["property_type"] == "New"]
print(new_only.groupby("vat_exclusive")["price"].describe()[["count", "25%", "50%", "75%"]].round(0).to_string())

                  count       25%       50%       75%
vat_exclusive                                        
No               2365.0   80000.0  158546.0  350000.0
Yes            140374.0  220264.0  308369.0  396476.0


2,365 sales are described as new dwellings but not flagged VAT-exclusive (1.7% of new builds). Their price distribution sits well below VAT-exclusive new builds rather than above, so these are unlikely to be gross-priced new sales. They are more plausibly non-standard transactions such as local authority or affordable housing transfers. No VAT adjustment is applied, consistent with keying the adjustment off the VAT flag rather than the property description.

### 2.4 Filtering

Three filters are applied.

Non-arms-length sales. The Not Full Market Price flag marks transfers between related parties, family transactions and similar. These are recorded prices but not market prices.

Price bounds of €20,000 to €2,000,000. This is a scoping decision rather than error correction. The upper tail contains genuine arms-length transactions that are not individual dwellings - entire apartment blocks, development sites and portfolio purchases, the largest being a €388m filing. The lower tail contains nominal transfers, including repeated exact values across unrelated properties. Only 14% of rows outside these bounds carry the non-market flag, confirming the bounds do independent work rather than duplicating it.

Placeholder Eircodes. Two dummy values are present. Since the sales themselves are genuine and only the Eircode is fabricated, the field is blanked rather than the row dropped.

In [13]:
n = len(ppr)

# 1. non-arms-length transactions
ppr = ppr[ppr["not_full_market"] == "No"].copy()
log_drop("Not full market price", n, len(ppr)); n = len(ppr)

# 2. restrict to individual dwellings
#    upper bound removes bulk and portfolio transactions
#    lower bound removes nominal transfers
PRICE_MIN, PRICE_MAX = 20_000, 2_000_000
ppr = ppr[ppr["price_incl_vat"].between(PRICE_MIN, PRICE_MAX)].copy()
log_drop(f"Price outside {PRICE_MIN:,} - {PRICE_MAX:,}", n, len(ppr)); n = len(ppr)

# 3. placeholder Eircodes - blank the code, keep the sale
PLACEHOLDERS = ["A123456", "A00AA00"]
mask = ppr["eircode"].isin(PLACEHOLDERS)
print(f"\nPlaceholder Eircodes blanked: {mask.sum():,}")
ppr.loc[mask, "eircode"] = np.nan

Not full market price                         dropped  40,721  remaining  758,346
Price outside 20,000 - 2,000,000              dropped   7,686  remaining  750,660

Placeholder Eircodes blanked: 154


In [14]:
survivors = ppr[(ppr["property_type"] == "New") & (ppr["vat_exclusive"] == "No")]
print(f"Remaining after filtering: {len(survivors):,}")

Remaining after filtering: 1,901


In [15]:
print(f"Rows now: {len(ppr):,}")
print(f"Drop log entries: {len(drop_log)}")

Rows now: 750,660
Drop log entries: 2


In [16]:
print(pd.DataFrame(drop_log).to_string(index=False))

                           stage  dropped  remaining  pct_of_original
           Not full market price    40721     758346             5.10
Price outside 20,000 - 2,000,000     7686     750660             1.01


### 2.5 Eircode validation and routing keys

A valid Eircode is a routing key of one letter and two digits, followed by a four-character unique identifier. Dublin 6W is the single exception, using a three-character routing key.

The routing key is the first three characters and identifies a geographic area rather than an individual property. Unlike full Eircodes, which require the licensed Eircode Address Database to interpret, routing keys are freely usable and provide sub-county granularity - particularly valuable in Dublin, where a single county contains widely varying submarkets.

In [17]:
# Dublin 6W is the one exception to the standard letter-digit-digit routing key format

EIRCODE_PATTERN = r"^([A-Za-z]\d{2}|D6W)\s?[A-Za-z0-9]{4}$"

ppr["eircode"] = ppr["eircode"].str.upper().str.replace(" ", "", regex=False)
valid = ppr["eircode"].str.match(EIRCODE_PATTERN, na=False)

print(f"Eircode present:  {ppr['eircode'].notna().sum():,}")
print(f"Format-valid:     {valid.sum():,}")
print(f"Malformed:        {(ppr['eircode'].notna() & ~valid).sum():,}")

# blank anything malformed rather than dropping the sale
ppr.loc[ppr["eircode"].notna() & ~valid, "eircode"] = np.nan

Eircode present:  229,756
Format-valid:     229,756
Malformed:        0


In [18]:
ppr["routing_key"] = ppr["eircode"].str[:3]

print(f"Sales with a routing key: {ppr['routing_key'].notna().sum():,}")
print(f"Distinct routing keys:    {ppr['routing_key'].nunique()}")
print(f"\nCoverage by year:")

cov = ppr.groupby("year").agg(
    sales=("routing_key", "size"),
    with_rk=("routing_key", "count"),
)
cov["pct"] = (cov["with_rk"] / cov["sales"] * 100).round(1)
print(cov.to_string())

Sales with a routing key: 229,756
Distinct routing keys:    302

Coverage by year:
      sales  with_rk   pct
year                      
2010  19777        1   0.0
2011  17215        2   0.0
2012  23411        0   0.0
2013  27888       21   0.1
2014  42047       52   0.1
2015  46870       38   0.1
2016  47461       65   0.1
2017  51848       97   0.2
2018  53674      105   0.2
2019  54821      161   0.3
2020  46455      309   0.7
2021  56480    29146  51.6
2022  59263    45764  77.2
2023  59187    45409  76.7
2024  56932    43074  75.7
2025  58257    43359  74.4
2026  29074    22153  76.2


### 2.6 Cross-county routing keys

Routing key areas are built around postal delivery patterns rather than administrative boundaries, so keys serving towns near a county border legitimately appear in both. Of 139 routing keys with at least 30 sales, median concentration in a single county is 0.995, but a minority are genuinely split - A92 covering Drogheda is 66% Louth and 34% Meath, and W91 around Naas and Blessington is 82% Kildare and 18% Wicklow.

Each routing key is therefore assigned its dominant county, providing a stable key for joining area-level BER data without misrepresenting the underlying geography.

In [19]:
# placeholder removal should substantially reduce apparent cross-county spread

xref = (ppr.dropna(subset=["routing_key"])
        .groupby("routing_key")["county"]
        .nunique()
        .sort_values(ascending=False))

print(f"Routing keys spanning multiple counties: {(xref > 1).sum()} of {len(xref)}")
print(f"\nMost cross-county:")
print(xref.head(15).to_string())

Routing keys spanning multiple counties: 142 of 302

Most cross-county:
routing_key
W91    14
V94    14
N41    12
H91    12
F91    11
N91    11
A96    11
W23    10
C15    10
R32     9
R93     8
F12     8
A94     8
R95     8
H12     8


In [20]:
rk = "W91"
sample = ppr[ppr["routing_key"] == rk]

print(f"{rk}: {len(sample):,} sales, {sample['eircode'].nunique():,} distinct Eircodes")
print(f"\nCounty spread:")
print(sample["county"].value_counts().to_string())
print(f"\nAddresses from the smaller counties:")
minor = sample["county"].value_counts().tail(8).index
print(sample[sample["county"].isin(minor)][["address", "county", "eircode"]].head(15).to_string())

W91: 4,350 sales, 4,135 distinct Eircodes

County spread:
county
Kildare      3566
Wicklow       764
Kilkenny        4
Dublin          4
Carlow          2
Donegal         2
Wexford         1
Cavan           1
Laois           1
Louth           1
Offaly          1
Westmeath       1
Clare           1
Meath           1

Addresses from the smaller counties:
                                               address     county  eircode
546738            COIS NA MARA, GRANGE, FETHARD ON SEA    Wexford  W91E1XK
567146                         URBAL, KILNALECK, CAVAN      Cavan  W91E299
634061  5 The Crescent, Mount Stewart, Stradbally Road      Laois  W91X8K3
648212              2 NEWTOWN PLACE, THE MEADOWS, KILL      Louth  W91F9PN
662351                       GRANGE, EDENDERRY, OFFALY     Offaly  W91PR64
673116     58 BURGAGE CASTLE, BLESSINGTON, CO. WICKLOW  Westmeath  W91H58A
715167            7 THE MILLICENT, COIS ABHAINN, CLANE      Clare  W91W324
725018                      KILTALE, DUNSANY,

### 2.7 County field errors

Inspection of low-volume routing key and county pairings reveals data entry errors in the county field. A Blessington, Co. Wicklow address is filed under Westmeath; a Clane address under Clare rather than Kildare. In each case the address text and the Eircode agree with each other and contradict the county field, indicating the county column is the unreliable one.

Pairings accounting for under 1% of a routing key's sales and fewer than ten transactions are flagged as suspect. These are flagged rather than deleted or reassigned - the sales are valid, and reassigning them to the dominant county would introduce a different error for legitimately cross-county routing keys. Modelling uses the routing key rather than the county field, so these errors do not affect model inputs.

In [21]:
# Routing key to dominant county 

# the county field contains occasional data entry errors, visible
# where the address text and Eircode both contradict it. Assigning
# each routing key its dominant county gives a stable mapping.

rk_county = (ppr.dropna(subset=["routing_key"])
             .groupby("routing_key")["county"]
             .agg(lambda s: s.mode().iat[0]))

ppr["rk_county"] = ppr["routing_key"].map(rk_county)

# how often does the filed county disagree with the routing key's dominant county?
has_rk = ppr["routing_key"].notna()
mismatch = has_rk & (ppr["county"] != ppr["rk_county"])
print(f"County disagrees with routing key's dominant county: {mismatch.sum():,} ({mismatch.sum()/has_rk.sum()*100:.1f}%)")

County disagrees with routing key's dominant county: 10,950 (4.8%)


In [22]:
# how concentrated is each routing key in its dominant county?
purity = (ppr.dropna(subset=["routing_key"])
          .groupby("routing_key")["county"]
          .agg(lambda s: s.value_counts(normalize=True).iat[0]))

print(purity.describe().round(3).to_string())
print(f"\nLeast concentrated routing keys:")
print(purity.nsmallest(15).round(3).to_string())

count    302.000
mean       0.953
std        0.124
min        0.286
25%        0.992
50%        1.000
75%        1.000
max        1.000

Least concentrated routing keys:
routing_key
A65    0.286
A00    0.500
A12    0.500
E15    0.500
E23    0.500
E42    0.500
F49    0.500
K91    0.500
N11    0.500
P91    0.500
T67    0.500
V12    0.500
X00    0.500
X95    0.500
Y94    0.500


In [23]:
rk_counts = ppr["routing_key"].value_counts()

low_purity = purity.nsmallest(20).index
print(pd.DataFrame({
    "purity": purity[low_purity].round(3),
    "n_sales": rk_counts[low_purity]
}).to_string())

             purity  n_sales
routing_key                 
A65           0.286        7
A00           0.500        2
A12           0.500        2
E15           0.500        2
E23           0.500        2
E42           0.500        2
F49           0.500        2
K91           0.500        2
N11           0.500        2
P91           0.500        2
T67           0.500        2
V12           0.500        2
X00           0.500        2
X95           0.500        2
Y94           0.500        2
A82           0.596     2022
A42           0.600       75
E32           0.636      522
A92           0.661     4625
N93           0.667        3


In [24]:
MIN_SALES = 30
reliable = purity[rk_counts.reindex(purity.index) >= MIN_SALES]

print(f"Routing keys with >= {MIN_SALES} sales: {len(reliable)} of {len(purity)}")
print(f"\nPurity distribution:")
print(reliable.describe().round(3).to_string())
print(f"\nLeast concentrated:")
print(pd.DataFrame({
    "purity": reliable.nsmallest(12).round(3),
    "n_sales": rk_counts[reliable.nsmallest(12).index]
}).to_string())

Routing keys with >= 30 sales: 139 of 302

Purity distribution:
count    139.000
mean       0.958
std        0.079
min        0.596
25%        0.958
50%        0.995
75%        0.998
max        1.000

Least concentrated:
             purity  n_sales
routing_key                 
A82           0.596     2022
A42           0.600       75
E32           0.636      522
A92           0.661     4625
F52           0.751      635
N37           0.756     2121
F91           0.802     3058
P51           0.817     2684
W91           0.820     4350
N41           0.833     1713
F26           0.848     1706
A81           0.854      513


In [25]:
# what does the A92 split actually look like?
a92 = ppr[ppr["routing_key"] == "A92"]
print(a92["county"].value_counts().to_string())

county
Louth      3059
Meath      1558
Dublin        5
Mayo          1
Cavan         1
Donegal       1


In [26]:
# Cross-county routing keys

# routing keys are built around postal delivery areas and legitimately
# straddle county boundaries where towns sit near a border

border_keys = reliable[reliable < 0.9].sort_values()
border = pd.DataFrame({
    "purity": border_keys.round(3),
    "n_sales": rk_counts[border_keys.index],
    "dominant_county": rk_county[border_keys.index],
})
print("Routing keys spanning counties (>=30 sales, purity <0.9):")
print(border.to_string())

Routing keys spanning counties (>=30 sales, purity <0.9):
             purity  n_sales dominant_county
routing_key                                 
A82           0.596     2022           Cavan
A42           0.600       75          Dublin
E32           0.636      522       Tipperary
A92           0.661     4625           Louth
F52           0.751      635       Roscommon
N37           0.756     2121       Westmeath
F91           0.802     3058           Sligo
P51           0.817     2684            Cork
W91           0.820     4350         Kildare
N41           0.833     1713         Leitrim
F26           0.848     1706            Mayo
A81           0.854      513        Monaghan
F45           0.855     1188       Roscommon
V94           0.855     8428        Limerick
P36           0.862      803            Cork
K32           0.863     1550          Dublin
F35           0.877      399            Mayo
R93           0.884     2498          Carlow
P56           0.886      438            Co

In [27]:
# How many county labels are likely typos? 

# within each routing key, counties accounting for a trivial share
# of sales are likely data entry errors rather than real geography

rk_county_counts = (ppr.dropna(subset=["routing_key"])
                    .groupby(["routing_key", "county"])
                    .size()
                    .rename("n")
                    .reset_index())

rk_totals = rk_county_counts.groupby("routing_key")["n"].transform("sum")
rk_county_counts["share"] = rk_county_counts["n"] / rk_totals

# a county with under 1% of a routing key's sales, and fewer than 10 sales,
# is very unlikely to be genuine geography
suspect = rk_county_counts[(rk_county_counts["share"] < 0.01) &
                           (rk_county_counts["n"] < 10)]

print(f"Suspect routing key / county pairs: {len(suspect):,}")
print(f"Sales affected: {suspect['n'].sum():,} "
      f"({suspect['n'].sum() / ppr['routing_key'].notna().sum() * 100:.2f}% of Eircoded sales)")

Suspect routing key / county pairs: 412
Sales affected: 611 (0.27% of Eircoded sales)


In [28]:
# flag rather than delete or overwrite
suspect_pairs = set(zip(suspect["routing_key"], suspect["county"]))
ppr["county_suspect"] = [
    (rk, c) in suspect_pairs
    for rk, c in zip(ppr["routing_key"], ppr["county"])
]
print(f"Flagged: {ppr['county_suspect'].sum():,}")

Flagged: 611


County field errors. Within routing keys, 611 sales (0.27% of Eircoded sales) are filed against a county accounting for under 1% of that routing key's transactions. Inspection confirms these are data entry errors rather than genuine geography - for example a Blessington, Co. Wicklow address filed under Westmeath, where both the address text and the Eircode contradict the county field. These rows are flagged via county_suspect rather than dropped or reassigned, since the sales themselves are valid and only one metadata field is wrong. Modelling uses the routing key rather than the county field, so these errors do not affect model inputs.

### 2.8 Output

The cleaned dataset is written to Parquet rather than CSV. Parquet preserves datetime and categorical types without re-parsing, loads faster, and produces a substantially smaller file - relevant here since the address column contains 750,000 free-text strings.

In [29]:
keep_cols = [
    "date", "year", "month", "address", "county",
    "eircode", "routing_key", "rk_county",
    "price", "price_incl_vat", "property_type",
    "not_full_market", "vat_exclusive", "county_suspect"
]

ppr_clean = ppr[keep_cols].copy()
ppr_clean.to_csv(PROCESSED / "ppr_clean.csv", index=False)

print(f"Saved {len(ppr_clean):,} rows to ppr_clean.csv")
print(f"\nColumns: {list(ppr_clean.columns)}")
print(f"\nMemory: {ppr_clean.memory_usage(deep=True).sum() / 1_048_576:.1f} MB")

Saved 750,660 rows to ppr_clean.csv

Columns: ['date', 'year', 'month', 'address', 'county', 'eircode', 'routing_key', 'rk_county', 'price', 'price_incl_vat', 'property_type', 'not_full_market', 'vat_exclusive', 'county_suspect']

Memory: 331.3 MB


In [30]:
size_mb = (PROCESSED / "ppr_clean.csv").stat().st_size / 1_048_576
print(f"ppr_clean.csv on disk: {size_mb:.1f} MB")

ppr_clean.csv on disk: 86.2 MB


In [31]:
# option 2 - parquet, smaller still and preserves dtypes
ppr_clean.to_parquet(PROCESSED / "ppr_clean.parquet", index=False)
print(f"parquet: {(PROCESSED / 'ppr_clean.parquet').stat().st_size / 1_048_576:.1f} MB")

parquet: 22.8 MB


## 3. SEAI BER Data

The BER export is tab-delimited, latin-1 encoded, and pads every text field with trailing spaces to a fixed width. The padding is stripped on load - left in place, "Detached house" and "Detached house " are treated as distinct categories and any grouping silently splits.

Of 252 columns, ten bear on property value. The remainder describe boiler efficiencies, U-values, solar collector properties and similar technical detail relevant to energy assessment but not to price.

In [32]:
# Load BER

ber = pd.read_csv(
    RAW / "BERPublicsearch.txt",
    sep="\t", encoding="latin-1", low_memory=False,
    on_bad_lines="skip", quoting=3
)

# strip the fixed-width padding from every text column
for col in ber.select_dtypes("object").columns:
    ber[col] = ber[col].str.strip()

n_ber_original = len(ber)
print(f"Loaded {n_ber_original:,} BER records")

Loaded 1,430,031 BER records


In [33]:
# Subset to relevant columns
# 10 of 252 columns bear on property value - the rest describe
# boiler efficiencies, U-values and similar technical detail

ber = ber[[
    "CountyName", "DwellingTypeDescr", "Year_of_Construction",
    "EnergyRating", "BerRating", "GroundFloorArea(sq m)",
    "FloorArea", "NoStoreys", "CO2Rating", "MainSpaceHeatingFuel",
]].rename(columns={
    "CountyName":            "ber_county",
    "DwellingTypeDescr":     "dwelling_type",
    "Year_of_Construction":  "year_built",
    "EnergyRating":          "energy_rating",
    "BerRating":             "ber_rating",
    "GroundFloorArea(sq m)": "ground_floor_area",
    "FloorArea":             "floor_area",
    "NoStoreys":             "n_storeys",
    "CO2Rating":             "co2_rating",
    "MainSpaceHeatingFuel":  "heating_fuel",
})

print(ber.dtypes.to_string())

ber_county            object
dwelling_type         object
year_built             int64
energy_rating         object
ber_rating           float64
ground_floor_area    float64
floor_area           float64
n_storeys              int64
co2_rating           float64
heating_fuel          object


### 3.1 Filtering implausible values

Notebook 01 identified impossible values across several fields - construction years as late as 2104, negative energy and CO2 ratings, floor areas exceeding 3,500 square metres, and dwellings recorded with zero storeys. Each is filtered to a plausible range.

All filters combined remove 5,839 of 1,430,031 records, retaining 99.6%. The largest single filter is energy rating at 0.37%.

In [34]:
# Filter implausible values 

n = len(ber)

ber = ber[ber["year_built"].between(1700, 2026)]
log_drop("BER: year of construction", n, len(ber)); n = len(ber)

ber = ber[ber["ber_rating"].between(0.01, 2000)]
log_drop("BER: energy rating", n, len(ber)); n = len(ber)

ber = ber[ber["co2_rating"] >= 0]
log_drop("BER: CO2 rating", n, len(ber)); n = len(ber)

ber = ber[ber["ground_floor_area"].between(10, 1000)]
log_drop("BER: ground floor area", n, len(ber)); n = len(ber)

ber = ber[ber["n_storeys"].between(1, 5)]
log_drop("BER: storeys", n, len(ber)); n = len(ber)

print(f"\nRetained {len(ber):,} of {n_ber_original:,} ({len(ber)/n_ber_original*100:.1f}%)")

BER: year of construction                     dropped       6  remaining 1,430,025
BER: energy rating                            dropped   5,287  remaining 1,424,738
BER: CO2 rating                               dropped     345  remaining 1,424,393
BER: ground floor area                        dropped      97  remaining 1,424,296
BER: storeys                                  dropped     104  remaining 1,424,192

Retained 1,424,192 of 1,430,031 (99.6%)


### 3.2 Resolving the floor area fields

The schema contains both GroundFloorArea(sq m) and FloorArea, and notebook 01 found the two inconsistent - a two-storey detached house showed 144.90 for the first and 90.29 for the second. Taking the column names at face value would mean the ground floor was larger than the whole dwelling.

Examining the ratio between the two fields by storey count resolves it. The median ratio is 1.00 for single-storey dwellings, 1.89 for two-storey and 2.34 for three-storey - tracking storey count almost exactly. Apartments show a FloorArea of zero while retaining a GroundFloorArea, consistent with having no ground floor footprint but a real total size.

The column names are therefore inverted relative to their contents. GroundFloorArea(sq m) holds total dwelling floor area and FloorArea holds the ground floor footprint. The fields are renamed accordingly, and total_floor_area is used for all size-based analysis.

This is worth flagging as a general caution: the schema alone would have led to systematically wrong size figures, particularly for apartments where the apparent floor area is zero.

In [35]:
# Which field represents total dwelling size? 
# in notebook 01 these two fields were inconsistent - a detached
# house showed ground_floor_area 179.59 and floor_area 90.29

sample = ber[["dwelling_type", "n_storeys", "ground_floor_area", "floor_area"]].head(20)
print(sample.to_string())

print("\nRatio of ground_floor_area to floor_area, by storey count:")
ber["area_ratio"] = ber["ground_floor_area"] / ber["floor_area"].replace(0, np.nan)
print(ber.groupby("n_storeys")["area_ratio"].describe()[["count", "25%", "50%", "75%"]].round(2).to_string())

           dwelling_type  n_storeys  ground_floor_area  floor_area
0         Detached house          2             144.90       90.29
1         Detached house          1             127.87      127.87
2      Mid-terrace house          2              94.78       49.17
3    Semi-detached house          2              92.89       48.71
4             Maisonette          2              71.96        0.00
5    Semi-detached house          2             138.80       69.40
6    Top-floor apartment          2              73.66        0.00
7         Detached house          2             199.67      164.04
8         Detached house          2              81.98       52.69
9    Semi-detached house          2             159.50       85.60
10  End of terrace house          2              63.68       31.84
11        Detached house          2             155.09       88.91
12        Detached house          2             276.80      155.40
13   Semi-detached house          2              93.26       4

In [36]:
# Resolve the floor area fields
# despite the column names, GroundFloorArea(sq m) contains total
# dwelling floor area and FloorArea contains the ground floor
# footprint. The ratio between them tracks storey count almost
# exactly - median 1.00 for single-storey, 1.89 for two-storey,
# 2.34 for three-storey - and apartments show a footprint of zero
# while retaining a total area.

ber = ber.rename(columns={
    "ground_floor_area": "total_floor_area",
    "floor_area":        "ground_floor_footprint",
})

ber = ber.drop(columns=["area_ratio"])

print(ber[["dwelling_type", "n_storeys",
           "total_floor_area", "ground_floor_footprint"]].head(10).to_string())

         dwelling_type  n_storeys  total_floor_area  ground_floor_footprint
0       Detached house          2            144.90                   90.29
1       Detached house          1            127.87                  127.87
2    Mid-terrace house          2             94.78                   49.17
3  Semi-detached house          2             92.89                   48.71
4           Maisonette          2             71.96                    0.00
5  Semi-detached house          2            138.80                   69.40
6  Top-floor apartment          2             73.66                    0.00
7       Detached house          2            199.67                  164.04
8       Detached house          2             81.98                   52.69
9  Semi-detached house          2            159.50                   85.60


In [37]:
# Aggregate BER to area level 
# BER cannot be joined to PPR per-property, since the public
# extract removes the address fields required. Area-level
# aggregates are joined instead.

ber_agg = ber.groupby("ber_county").agg(
    ber_n=("ber_rating", "size"),
    mean_floor_area=("total_floor_area", "mean"),
    median_floor_area=("total_floor_area", "median"),
    mean_ber_rating=("ber_rating", "mean"),
    median_year_built=("year_built", "median"),
    mean_co2=("co2_rating", "mean"),
    mean_storeys=("n_storeys", "mean"),
).round(2)

print(f"Areas: {len(ber_agg)}")
print(ber_agg.sort_values("ber_n", ascending=False).head(20).to_string())

Areas: 55
                ber_n  mean_floor_area  median_floor_area  mean_ber_rating  median_year_built  mean_co2  mean_storeys
ber_county                                                                                                           
Co. Cork       137433           124.61             108.42           197.05             1998.0     44.80          1.82
Co. Dublin     114115           115.89             105.15           157.33             2002.0     32.05          1.86
Co. Kildare     68592           124.42             112.43           159.14             2003.0     35.46          1.81
Co. Meath       58339           131.23             116.65           161.03             2004.0     36.45          1.82
Co. Galway      52069           142.06             123.11           220.09             1999.0     54.52          1.73
Co. Wexford     47383           130.06             111.90           211.05             2000.0     51.79          1.74
Co. Wicklow     46318           123.97        

In [38]:
# proportion of dwellings rated A or B, a useful area-level
# indicator of housing stock quality
ber["is_ab"] = ber["energy_rating"].str[0].isin(["A", "B"])

ab_share = ber.groupby("ber_county")["is_ab"].mean().round(3).rename("pct_a_or_b")
ber_agg = ber_agg.join(ab_share)

print(ber_agg.sort_values("pct_a_or_b", ascending=False).head(15).to_string())

              ber_n  mean_floor_area  median_floor_area  mean_ber_rating  median_year_built  mean_co2  mean_storeys  pct_a_or_b
ber_county                                                                                                                     
Dublin 18     21771           108.53              89.07           112.03             2008.0     21.57          1.59       0.709
Co. Meath     58339           131.23             116.65           161.03             2004.0     36.45          1.82       0.515
Co. Kildare   68592           124.42             112.43           159.14             2003.0     35.46          1.81       0.512
Co. Dublin   114115           115.89             105.15           157.33             2002.0     32.05          1.86       0.512
Dublin 13     13307           104.97              93.22           157.90             2003.0     31.27          1.78       0.510
Dublin 15     33347           104.52              96.11           150.83             2003.0     29.74   

### 3.3 Joining BER to PPR

The two datasets share no common key. The PPR records addresses and Eircodes; the BER research extract removes both, since a BER assessment includes the property address and MPRN and is therefore personal data under GDPR. A per-property join is not possible from public data.

Area-level aggregation is used instead. BER area names take three forms - Co. Cork, Dublin 15 and Limerick City - which are normalised into a county plus optional Dublin district. On the PPR side, Dublin postal districts are recovered from Eircode routing keys via an explicit whitelist of D01 to D24 plus D6W.

A whitelist rather than pattern matching is necessary because a leading D does not indicate Dublin. Routing keys such as D93 and D95 appear in the data filed against Donegal and Clare addresses - these are typos in the Eircode rather than Dublin properties, twelve rows in total, and are excluded from district assignment.

The resulting join operates at postal district level within Dublin and county level elsewhere. Cork, Galway, Limerick and Waterford cities fold into their counties, since the PPR makes no city and county distinction. Dublin sales without an Eircode fall back to Dublin-wide averages.

In [39]:
# What do the two datasets call things?

print("BER area names:")
print(sorted(ber["ber_county"].unique()))

print("\nPPR counties:")
print(sorted(ppr_clean["county"].unique()))

BER area names:
['Co. Carlow', 'Co. Cavan', 'Co. Clare', 'Co. Cork', 'Co. Donegal', 'Co. Dublin', 'Co. Galway', 'Co. Kerry', 'Co. Kildare', 'Co. Kilkenny', 'Co. Laois', 'Co. Leitrim', 'Co. Limerick', 'Co. Longford', 'Co. Louth', 'Co. Mayo', 'Co. Meath', 'Co. Monaghan', 'Co. Offaly', 'Co. Roscommon', 'Co. Sligo', 'Co. Tipperary', 'Co. Waterford', 'Co. Westmeath', 'Co. Wexford', 'Co. Wicklow', 'Cork City', 'Dublin 1', 'Dublin 10', 'Dublin 11', 'Dublin 12', 'Dublin 13', 'Dublin 14', 'Dublin 15', 'Dublin 16', 'Dublin 17', 'Dublin 18', 'Dublin 19', 'Dublin 2', 'Dublin 20', 'Dublin 21', 'Dublin 22', 'Dublin 23', 'Dublin 24', 'Dublin 3', 'Dublin 4', 'Dublin 5', 'Dublin 6', 'Dublin 6W', 'Dublin 7', 'Dublin 8', 'Dublin 9', 'Galway City', 'Limerick City', 'Waterford City']

PPR counties:
['Carlow', 'Cavan', 'Clare', 'Cork', 'Donegal', 'Dublin', 'Galway', 'Kerry', 'Kildare', 'Kilkenny', 'Laois', 'Leitrim', 'Limerick', 'Longford', 'Louth', 'Mayo', 'Meath', 'Monaghan', 'Offaly', 'Roscommon', 'Sligo

In [40]:
# Normalise BER area names 

def parse_ber_area(name):
    """Return (county, dublin_district) for a BER area name."""
    n = name.strip()
    if n.startswith("Co. "):
        return n[4:], None
    if n.startswith("Dublin ") and n[7:].strip()[0].isdigit():
        return "Dublin", n
    if n.endswith(" City"):
        return n[:-5], None
    if n == "Co. Dublin":
        return "Dublin", None
    return n, None

parsed = ber["ber_county"].map(parse_ber_area)
ber["county_norm"] = [p[0] for p in parsed]
ber["dublin_district"] = [p[1] for p in parsed]

print(f"Normalised counties: {ber['county_norm'].nunique()}")
print(sorted(ber["county_norm"].unique()))
print(f"\nDublin districts: {ber['dublin_district'].nunique()}")

Normalised counties: 26
['Carlow', 'Cavan', 'Clare', 'Cork', 'Donegal', 'Dublin', 'Galway', 'Kerry', 'Kildare', 'Kilkenny', 'Laois', 'Leitrim', 'Limerick', 'Longford', 'Louth', 'Mayo', 'Meath', 'Monaghan', 'Offaly', 'Roscommon', 'Sligo', 'Tipperary', 'Waterford', 'Westmeath', 'Wexford', 'Wicklow']

Dublin districts: 25


In [41]:
# Dublin routing key to postal district 
# D01 -> Dublin 1, D6W -> Dublin 6W, and so on

def dublin_district_from_rk(rk):
    if not isinstance(rk, str) or not rk.startswith("D"):
        return None
    suffix = rk[1:]
    if suffix == "6W":
        return "Dublin 6W"
    if suffix.isdigit():
        return f"Dublin {int(suffix)}"
    return None

ppr_clean["dublin_district"] = ppr_clean["routing_key"].map(dublin_district_from_rk)

print(f"Sales with a Dublin district: {ppr_clean['dublin_district'].notna().sum():,}")
print(ppr_clean["dublin_district"].value_counts().head(20).to_string())

Sales with a Dublin district: 59,338
dublin_district
Dublin 15    6656
Dublin 24    4629
Dublin 18    4127
Dublin 8     3729
Dublin 4     3353
Dublin 13    3220
Dublin 9     3189
Dublin 7     3166
Dublin 12    3073
Dublin 3     2773
Dublin 14    2762
Dublin 11    2717
Dublin 16    2657
Dublin 5     2315
Dublin 6     2275
Dublin 22    2265
Dublin 1     1728
Dublin 6W    1490
Dublin 2     1140
Dublin 10     742


In [42]:
ber_districts = set(ber["dublin_district"].dropna().unique())
ppr_districts = set(ppr_clean["dublin_district"].dropna().unique())

print(f"BER districts: {len(ber_districts)}")
print(f"PPR districts: {len(ppr_districts)}")
print(f"\nIn BER not PPR: {sorted(ber_districts - ppr_districts)}")
print(f"In PPR not BER: {sorted(ppr_districts - ber_districts)}")

BER districts: 25
PPR districts: 32

In BER not PPR: ['Dublin 21', 'Dublin 23']
In PPR not BER: ['Dublin 25', 'Dublin 27', 'Dublin 56', 'Dublin 62', 'Dublin 67', 'Dublin 91', 'Dublin 93', 'Dublin 94', 'Dublin 95']


In [43]:
fake = ["D25", "D27", "D56", "D62", "D67", "D91", "D93", "D94", "D95"]
chk = ppr_clean[ppr_clean["routing_key"].isin(fake)]
print(chk.groupby("routing_key").agg(
    n=("county", "size"),
    counties=("county", lambda s: s.value_counts().index[:3].tolist())
).to_string())

             n   counties
routing_key              
D25          2   [Dublin]
D27          1   [Dublin]
D56          1  [Kildare]
D62          1   [Galway]
D67          1   [Dublin]
D91          1   [Galway]
D93          1  [Donegal]
D94          2  [Donegal]
D95          2    [Clare]


In [44]:
print(sorted([rk for rk in ppr_clean["routing_key"].dropna().unique() if rk.startswith("D")]))

['D01', 'D02', 'D03', 'D04', 'D05', 'D06', 'D07', 'D08', 'D09', 'D10', 'D11', 'D12', 'D13', 'D14', 'D15', 'D16', 'D17', 'D18', 'D19', 'D20', 'D22', 'D24', 'D25', 'D27', 'D56', 'D62', 'D67', 'D6W', 'D91', 'D93', 'D94', 'D95']


In [45]:
# Dublin routing key to postal district 
# Dublin postal districts are 1-24 plus 6W. Routing keys beginning
# with D outside that range are not Dublin - the letter prefix is
# simply the first character of an area code, not a county marker.

DUBLIN_DISTRICTS = {f"D{i:02d}": f"Dublin {i}" for i in range(1, 25)}
DUBLIN_DISTRICTS["D6W"] = "Dublin 6W"

ppr_clean["dublin_district"] = ppr_clean["routing_key"].map(DUBLIN_DISTRICTS)

print(f"Sales with a Dublin district: {ppr_clean['dublin_district'].notna().sum():,}")
print(f"Districts: {ppr_clean['dublin_district'].nunique()}")

Sales with a Dublin district: 59,326
Districts: 23


In [46]:
# Build a common join key

ber["join_area"] = ber["dublin_district"].fillna(ber["county_norm"])
ppr_clean["join_area"] = ppr_clean["dublin_district"].fillna(ppr_clean["county"])

print(f"BER join areas: {ber['join_area'].nunique()}")
print(f"PPR join areas: {ppr_clean['join_area'].nunique()}")
print(f"\nIn PPR not BER: {sorted(set(ppr_clean['join_area']) - set(ber['join_area']))}")

BER join areas: 51
PPR join areas: 49

In PPR not BER: []


In [47]:
# Re-aggregate BER on the join key 

ber["is_ab"] = ber["energy_rating"].str[0].isin(["A", "B"])

ber_agg = ber.groupby("join_area").agg(
    ber_n=("ber_rating", "size"),
    ber_mean_floor_area=("total_floor_area", "mean"),
    ber_median_floor_area=("total_floor_area", "median"),
    ber_mean_rating=("ber_rating", "mean"),
    ber_median_year_built=("year_built", "median"),
    ber_mean_co2=("co2_rating", "mean"),
    ber_mean_storeys=("n_storeys", "mean"),
    ber_pct_a_or_b=("is_ab", "mean"),
).round(3)

print(f"Areas: {len(ber_agg)}")
print(ber_agg.sort_values("ber_n", ascending=False).head(10).to_string())

Areas: 51
            ber_n  ber_mean_floor_area  ber_median_floor_area  ber_mean_rating  ber_median_year_built  ber_mean_co2  ber_mean_storeys  ber_pct_a_or_b
join_area                                                                                                                                            
Cork       157378              120.742                 104.90          202.635                 1997.0        45.613             1.827           0.399
Dublin     114115              115.895                 105.15          157.333                 2002.0        32.051             1.856           0.512
Galway      74008              131.958                 114.05          214.656                 1999.0        51.742             1.758           0.326
Kildare     68592              124.422                 112.43          159.143                 2003.0        35.457             1.813           0.512
Meath       58339              131.227                 116.65          161.033            

In [48]:
# Join to PPR 

before = len(ppr_clean)
ppr_joined = ppr_clean.merge(ber_agg, on="join_area", how="left")

print(f"Rows before: {before:,}  after: {len(ppr_joined):,}")
print(f"Unmatched:   {ppr_joined['ber_n'].isna().sum():,}")

Rows before: 750,660  after: 750,660
Unmatched:   0


The join adds eight area-level BER features to every sale: mean and median floor area, mean energy rating, median construction year, mean CO2 rating, mean storey count, the proportion of dwellings rated A or B, and the number of BER records underlying each area average. All 750,660 sales match an area.

One caveat on interpretation. A BER assessment is only required at point of sale or rental, so the dataset over-represents recently transacted and newly built stock. ber_pct_a_or_b measures the efficiency of assessed dwellings in an area, not of the housing stock as a whole.

In [49]:
# Save 

ppr_joined.to_parquet(PROCESSED / "ppr_with_ber.parquet", index=False)
ber.to_parquet(PROCESSED / "ber_clean.parquet", index=False)
ber_agg.to_csv(PROCESSED / "ber_area_aggregates.csv")

print(f"ppr_with_ber.parquet:      {len(ppr_joined):,} rows")
print(f"ber_clean.parquet:         {len(ber):,} rows")
print(f"ber_area_aggregates.csv:   {len(ber_agg)} areas")

ppr_with_ber.parquet:      750,660 rows
ber_clean.parquet:         1,424,192 rows
ber_area_aggregates.csv:   51 areas


## 4. Cleaning Summary
Outputs
File	Rows	Contents
ppr_clean.parquet	750,660	Cleaned sales, 2010 to present
ppr_with_ber.parquet	750,660	As above with area-level BER features joined
ber_clean.parquet	1,424,192	Cleaned BER records
ber_area_aggregates.csv	51	Area-level BER summary
Key decisions

VAT adjustment applied to 140,374 new build sales at 13.5%, keyed off the VAT flag rather than the property description.

Price bounds of €20,000 to €2,000,000 scope the dataset to individual dwellings, removing bulk transactions and nominal transfers. This is a scoping decision, not error correction - the excluded high-value sales are genuine arms-length transactions of apartment blocks and development sites.

Routing key is the primary geographic unit, not county. The county field contains data entry errors visible where both the address text and the Eircode contradict it, while Eircodes are validated at filing. 611 sales carry a county_suspect flag.

BER joins at area level, since GDPR anonymisation of the public extract makes a per-property join impossible.

Next

Notebook 03 covers exploratory analysis: price trends by county and routing key, new versus second-hand, seasonality, and the relationship between area energy efficiency and price.

In [50]:
# Full cleaning summary

print(pd.DataFrame(drop_log).to_string(index=False))

                           stage  dropped  remaining  pct_of_original
           Not full market price    40721     758346             5.10
Price outside 20,000 - 2,000,000     7686     750660             1.01
       BER: year of construction        6    1430025             0.00
              BER: energy rating     5287    1424738             0.37
                 BER: CO2 rating      345    1424393             0.02
          BER: ground floor area       97    1424296             0.01
                    BER: storeys      104    1424192             0.01
